In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
!pip install pandas  pathlib biopython matplotlib
import pandas
import re
import sys
from pathlib import Path
import pandas as pd
from ete3 import NCBITaxa
import matplotlib.pyplot as plt

ncbi = NCBITaxa()

analysis_dir = Path("/home/saj/jiss/barbeque/analysis")
# The common files for all primers
teeliste = analysis_dir / "teeliste.tsv"
taxids_db = analysis_dir / "build_db_taxids/null.db_taxids_counts.tsv"
# Find all consensus files across all primers
consensus_files = list((analysis_dir / "consensus").glob("*.cluster_consensus.tsv"))
all_results = []


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 12.2 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 19.2 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 9.7 MB/s  0:00:0036m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [matplotlib]8 [matplotlib]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [8]:
teeliste_df = pd.read_csv(teeliste, sep="\t", header=0)
teeliste_df.columns = ["german_name", "latin_name", "taxid"]

taxids_db_df = pd.read_csv(taxids_db, sep="\t", header=None)
taxids_db_df.columns = ["taxid", "count"]

taxids = teeliste_df["taxid"].astype(str).str.strip()

all_results = []

for consensus in consensus_files:
    primer_prefix = consensus.name.replace(".cluster_consensus.tsv", "")
    taxonomy = analysis_dir / f"join_accession_taxonomy/{primer_prefix}.cluster_taxonomy.tsv"
    amplicon_lengths = analysis_dir / f"amplicon_lengths/{primer_prefix}.amplicon_lengths.tsv"

    if not (taxonomy.exists() and amplicon_lengths.exists()):
        print(f"Skipping {primer_prefix} - Missing required files.")
        continue

    df = pd.read_csv(consensus, sep="\t", header=None)
    df.columns = ["cluster_id", "accession", "accession_taxid", "accession_name",
                  "assigned_name", "assigned_taxid", "assigned_rank", "cluster_names"]

    amplicon_lengths_df = pd.read_csv(amplicon_lengths, sep="\t", header=None)
    amplicon_lengths_df.columns = ["description", "lengths"]

    accession_counts = (
        df["accession_taxid"]
        .astype(str)
        .str.strip()
        .value_counts()
    )
    assigned_counts = (
        df[["cluster_id", "assigned_taxid"]]
        .astype(str)
        .apply(lambda col: col.str.strip())
        .drop_duplicates()
        .groupby("assigned_taxid")["cluster_id"]
        .nunique()
    )
    assigned_rank_map = (
        df[["assigned_taxid", "assigned_rank"]]
        .astype(str)
        .apply(lambda col: col.str.strip())
        .groupby("assigned_taxid")["assigned_rank"]
        .agg(lambda ranks: ";".join(sorted(set(ranks))))
    )
    assigned_name_map = (
        df[["assigned_taxid", "assigned_name"]]
        .astype(str)
        .apply(lambda col: col.str.strip())
        .groupby("assigned_taxid")["assigned_name"]
        .agg(lambda names: ";".join(sorted(set(names))))
    )
    db_count_map = (
        taxids_db_df[["taxid", "count"]]
        .astype(str)
        .apply(lambda col: col.str.strip())
        .groupby("taxid")["count"]
        .agg(lambda counts: ";".join(sorted(set(counts))))
    )

    accession_to_assigned_name_map = (
        df[["accession_taxid", "assigned_name"]]
        .astype(str)
        .apply(lambda col: col.str.strip())
        .groupby("accession_taxid")["assigned_name"]
        .agg(lambda names: ";".join(sorted(set(names))))
    )

    accession_to_assigned_taxid_map = (
        df[["accession_taxid", "assigned_taxid"]]
        .astype(str)
        .apply(lambda col: col.str.strip())
        .groupby("accession_taxid")["assigned_taxid"]
        .agg(lambda ids: ";".join(sorted(set(ids))))
    )

    accession_to_assigned_rank_map = (
        df[["accession_taxid", "assigned_rank"]]
        .astype(str)
        .apply(lambda col: col.str.strip())
        .groupby("accession_taxid")["assigned_rank"]
        .agg(lambda ranks: ";".join(sorted(set(ranks))))
    )

    # 1. Extract accession from the amplicon length file
    amplicon_lengths_df["accession"] = (
        amplicon_lengths_df["description"]
        .astype(str)
        .str.split()
        .str[0]
        .str.replace(r"\.\d+$", "", regex=True)
    )

    # 2. Clean accession in consensus table
    df["accession"] = (
        df["accession"]
        .astype(str)
        .str.replace(r"\.\d+$", "", regex=True)
    )

    # Create accession -> taxid map
    accession_taxid_map = (
        df[["accession", "accession_taxid"]]
        .astype(str)
        .apply(lambda col: col.str.strip())
        .drop_duplicates()
        .set_index("accession")["accession_taxid"]
    )

    # Add accession_taxid directly to amplicon_lengths_df
    amplicon_lengths_df["lengths"] = (
        pd.to_numeric(amplicon_lengths_df["lengths"], errors="coerce")
        .astype("Int64")
    )

    amplicon_lengths_df["accession_taxid"] = (
        amplicon_lengths_df["accession"]
        .map(accession_taxid_map)
    )

    # Summarize lengths per taxid
    length_summary = (
        amplicon_lengths_df
        .dropna(subset=["accession_taxid", "lengths"])
        .groupby("accession_taxid")["lengths"]
        .agg(["count", "min", "mean", "median", "max"])
    ).round(0).astype("Int64")

    lengths_list_by_taxid = (
        amplicon_lengths_df
        .dropna(subset=["accession_taxid", "lengths"])
        .groupby("accession_taxid")["lengths"]
        .agg(lambda x: ";".join(map(str, sorted(x.dropna().astype(int)))))
    )

    results_df = pd.DataFrame({
        "taxid": taxids,
        "german_name": teeliste_df["german_name"],
        "latin_name": teeliste_df["latin_name"],
        "no_of_occurance": taxids.map(accession_counts).fillna(0).astype(int),
        "LCA_which resolved_ to_this_taxid": taxids.map(assigned_counts).fillna(0).astype(int),
        "name_of_the_lca_taxid": taxids.map(assigned_name_map).fillna("not_assigned"),
        "name_of_the_lca_rank": taxids.map(assigned_rank_map).fillna("NA"),
        "consensus_name": taxids.map(accession_to_assigned_name_map).fillna("not_found"),
        "consensus_taxid": taxids.map(accession_to_assigned_taxid_map).fillna("not_found"),
        "consensus_rank": taxids.map(accession_to_assigned_rank_map).fillna("not_found"),
        "db_count": taxids.map(db_count_map).fillna("0"),
    })

    results_df["amplified"] = results_df["no_of_occurance"].apply(lambda x: "yes" if x > 0 else "no")
    results_df["taxid"] = results_df["taxid"].astype(str).str.strip()

    length_summary.index = length_summary.index.astype(str).str.strip()
    results_df = results_df.merge(length_summary, left_on="taxid", right_index=True, how="left")
    results_df["amplicon_lengths"] = results_df["taxid"].map(lengths_list_by_taxid).fillna("not_found")

    results_df["primer"] = primer_prefix
    all_results.append(results_df)

final_df = pd.concat(all_results, ignore_index=True)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.expand_frame_repr", False)
print(final_df)

       taxid              german_name              latin_name  no_of_occurance  LCA_which resolved_ to_this_taxid                name_of_the_lca_taxid name_of_the_lca_rank                                     consensus_name              consensus_taxid                 consensus_rank db_count amplified  count   min  mean  median   max                                   amplicon_lengths                                             primer
0       4442                      Tee       Camellia sinensis                0                                  0                         not_assigned                   NA                                          not_found                    not_found                      not_found   105179        no   <NA>  <NA>  <NA>    <NA>  <NA>                                          not_found                                    PCR_ITS1_custom
1     271192                     Anis       Pimpinella anisum                0                                  0             

In [23]:
species_resolution_detail = (
    final_df[final_df["consensus_rank"].str.contains("species", na=False)]
    .groupby("primer")[["taxid", "german_name", "latin_name"]]
    .apply(lambda g: g.drop_duplicates(subset="taxid")[["german_name", "latin_name"]])
)
print(species_resolution_detail)

                                                              german_name              latin_name
primer                                                                                           
ITS2_collapsed_custom                              740                Tee       Camellia sinensis
                                                   742              Apfel         Malus domestica
                                                   743              Apfel        Malus sylvestris
                                                   758         Grapefruit         Citrus paradisi
                                                   770             Jasmin     Jasminum officinale
                                                   773     Katzenpfötchen   Helichrysum arenarium
                                                   777     Limette/Limone  Citrus x aurantiifolia
                                                   780              Linde         Tilia tomentosa
                    

In [24]:
fully_resolved_detail = (
    final_df[final_df["consensus_rank"] == "species"]
    .drop_duplicates(subset=["primer", "taxid"])
    [["primer", "german_name", "latin_name", "consensus_name", "taxid"]]
    .sort_values(["primer", "latin_name"])
)
print(fully_resolved_detail)

                                                primer        german_name             latin_name         consensus_name   taxid
773                              ITS2_collapsed_custom     Katzenpfötchen  Helichrysum arenarium  Helichrysum arenarium  261776
809                              ITS2_collapsed_custom               Mate    Ilex paraguariensis    Ilex paraguariensis  185542
770                              ITS2_collapsed_custom             Jasmin    Jasminum officinale    Jasminum officinale  126433
434    Internal-transcribed-spacer-region-2-480_custom    Zitronenverbene     Aloysia citriodora      Aloysia citrodora  925377
370    Internal-transcribed-spacer-region-2-480_custom                Tee      Camellia sinensis      Camellia sinensis    4442
404    Internal-transcribed-spacer-region-2-480_custom          Kornblume       Centaurea cyanus       Centaurea cyanus   41522
439    Internal-transcribed-spacer-region-2-480_custom               Mate    Ilex paraguariensis    Ilex

In [12]:
coverage = (
    final_df.groupby("primer")["amplified"]
    .apply(lambda x: (x == "yes").sum())
    .sort_values(ascending=False)
)
print(coverage)

primer
Rubisco-ribulose-1,5-bisphosphate-carboxylase_oxygenase-large-subunit-180_custom    67
intron-region-of-a-transfer-RNA-gene-133-tnrl_custom                                60
ITS2_collapsed_custom                                                               60
intron-region-of-a-transfer-RNA-gene-200-500_custom                                 58
PCR_trnL_custom                                                                     58
Internal-transcribed-spacer-region-2-480_custom                                     49
PCR_psbA-trnH_custom                                                                42
PCR_ITS1_custom                                                                     35
PCR_ITS2_custom                                                                     35
PCR_rbcL_custom                                                                      0
PCR_matK_custom                                                                      0
Name: amplified, dtype: int64


In [13]:
coverage_pct = (
    final_df.groupby("primer")["amplified"]
    .apply(lambda x: (x == "yes").mean() * 100)
    .round(1)
    .sort_values(ascending=False)
)
print(coverage_pct)

primer
Rubisco-ribulose-1,5-bisphosphate-carboxylase_oxygenase-large-subunit-180_custom    90.5
intron-region-of-a-transfer-RNA-gene-133-tnrl_custom                                81.1
ITS2_collapsed_custom                                                               81.1
intron-region-of-a-transfer-RNA-gene-200-500_custom                                 78.4
PCR_trnL_custom                                                                     78.4
Internal-transcribed-spacer-region-2-480_custom                                     66.2
PCR_psbA-trnH_custom                                                                56.8
PCR_ITS1_custom                                                                     47.3
PCR_ITS2_custom                                                                     47.3
PCR_rbcL_custom                                                                      0.0
PCR_matK_custom                                                                      0.0
Name: amplifie

In [14]:
summary = final_df.groupby("primer").agg(
    coverage_pct=("amplified", lambda x: (x == "yes").mean() * 100),
    species_resolved_pct=("consensus_rank", lambda x: (x == "species").mean() * 100),
).round(1).sort_values("species_resolved_pct", ascending=False)
print(summary)

                                                    coverage_pct  species_resolved_pct
primer                                                                                
PCR_ITS2_custom                                             47.3                  12.2
PCR_psbA-trnH_custom                                        56.8                  10.8
Internal-transcribed-spacer-region-2-480_custom             66.2                  10.8
PCR_ITS1_custom                                             47.3                   6.8
ITS2_collapsed_custom                                       81.1                   4.1
intron-region-of-a-transfer-RNA-gene-200-500_cu...          78.4                   1.4
PCR_trnL_custom                                             78.4                   1.4
PCR_rbcL_custom                                              0.0                   0.0
PCR_matK_custom                                              0.0                   0.0
Rubisco-ribulose-1,5-bisphosphate-carboxyla

In [ ]:
!pip install openpyxl
final_df.to("/home/saj/jiss/barbeque/analysis/primer_results_all.xlsx", index=False)

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


checking the occurance of the accession taxid or the assigned taxid in the cluster_consensus

### Results Summary Dataframe
This step maps the target taxa (from `teeliste.tsv`) to the consensus results. The resulting dataframe contains the following key columns:

**1. Target Information**
- `taxid` / `german_name` / `latin_name`: Target metadata extracted from the input list (`teeliste.tsv`).

**2. Sequence & Cluster Abundance**
- `sequences_with_target_taxid`: Total raw accessions matching this target `taxid` before clustering.
- `clusters_resolved_to_target`: Number of final clusters resolving to this target `taxid`.

**3. Actual Consensus Tracking**
- `actual_consensus_name` / `taxid` / `rank`: Identifies what cluster the target sequences *actually* resolved to. Useful for detecting when species collapse to genus- or family-level.
- `resolved_name` / `resolved_rank`: The name and rank assigned when it matched the target.

**4. Database Context**
- `sequences_in_database`: Total reference sequences available in the Blast database for this `taxid`.

**5. Amplicon Length Statistics**
- `amplicon_count`, `min_length`, `mean_length`, `median_length`, `max_length`: Length statistics of amplicons mapped to this `taxid`.
- `amplicon_lengths`: Semicolon-separated string of exact lengths.


In [25]:
its2_amplify = (
    final_df[final_df["primer"].isin(["ITS2_collapsed_custom", ])]
    .groupby("primer")["amplified"]
    .apply(lambda x: (x == "yes").sum())
)
print(its2_amplify)

primer
ITS2_collapsed_custom    60
Name: amplified, dtype: int64


In [29]:
rbcl = final_df[final_df["primer"] == "Rubisco-ribulose-1,5-bisphosphate-carboxylase_oxygenase-large-subunit-180_custom"]
print((rbcl["amplified"] == "yes").sum(), "out of", len(rbcl))

67 out of 74


In [30]:
tnrl = final_df[final_df["primer"] == "intron-region-of-a-transfer-RNA-gene-133-tnrl_custom"]
print((tnrl["amplified"] == "yes").sum(), "out of", len(tnrl))

60 out of 74


In [ ]:
import pandas as pd
import plotly.express as px

# -----------------------------
# Clean dataframe
# -----------------------------

plot_df = plot_df.copy()


# Remove accidental header row
plot_df = plot_df[
    plot_df["taxid"].astype(str).str.strip() != "taxid"
].copy()

# Clean text columns
for col in [
    "taxid",
    "german_name",
    "latin_name",
    "consensus_taxid",
    "consensus_name",
    "consensus_rank",
    "amplified",
    "name_of_the_lca_taxid",
    "name_of_the_lca_rank",
]:
    if col in plot_df.columns:
        plot_df[col] = plot_df[col].astype(str).str.strip()

# Numeric columns
plot_df["no_of_occurance"] = pd.to_numeric(
    plot_df["no_of_occurance"], errors="coerce"
).fillna(0)

plot_df["LCA_which resolved_ to_this_taxid"] = pd.to_numeric(
    plot_df["LCA_which resolved_ to_this_taxid"], errors="coerce"
).fillna(0)

plot_df["db_count"] = pd.to_numeric(
    plot_df["db_count"], errors="coerce"
).fillna(0)


# -----------------------------
# Helper functions
# -----------------------------

def split_values(value):
    value = str(value).strip()
    if value in ["", "nan", "None", "NA", "not_found", "not_assigned"]:
        return []
    return [x.strip() for x in value.split(";") if x.strip()]


def classify_resolution(row):
    taxid = str(row["taxid"]).strip()
    consensus_taxids = split_values(row["consensus_taxid"])
    consensus_ranks = split_values(row["consensus_rank"])
    no_of_occurance = row["no_of_occurance"]
    direct_lca_count = row["LCA_which resolved_ to_this_taxid"]

    # No amplicon/accession evidence
    if no_of_occurance == 0 and direct_lca_count == 0:
        return "Not amplified / no evidence"

    # Fully resolved: only one consensus taxid and it is the target taxid
    if len(consensus_taxids) == 1 and consensus_taxids[0] == taxid:
        return "Fully resolved"

    # Partially resolved: target taxid appears, but with other broader/mixed LCAs
    if taxid in consensus_taxids and len(consensus_taxids) > 1:
        return "Partially resolved / mixed"

    # Genus-level only
    if "genus" in consensus_ranks:
        return "Genus-level only"

    # Family-level only
    if "family" in consensus_ranks:
        return "Family-level only"

    # Higher-level collapse
    higher_ranks = {
        "order",
        "class",
        "clade",
        "tribe",
        "subfamily",
        "subtribe",
        "subgenus",
        "subsection",
    }

    if any(rank in higher_ranks for rank in consensus_ranks):
        return "Higher-rank collapse"

    # If direct LCA exists but no source occurrence
    if direct_lca_count > 0 and no_of_occurance == 0:
        return "Resolved as LCA target only"

    return "Ambiguous / check manually"


def count_mixed_taxids(value):
    return len(split_values(value))


def count_mixed_ranks(value):
    return len(split_values(value))


# -----------------------------
# Add classification columns
# -----------------------------

plot_df["resolution_status"] = plot_df.apply(classify_resolution, axis=1)

plot_df["number_of_consensus_taxids"] = plot_df["consensus_taxid"].apply(count_mixed_taxids)
plot_df["number_of_consensus_ranks"] = plot_df["consensus_rank"].apply(count_mixed_ranks)

plot_df["is_taxid_mixed"] = plot_df["number_of_consensus_taxids"] > 1

plot_df["taxon_label"] = (
    plot_df["german_name"].fillna("NA")
    + " / "
    + plot_df["latin_name"].fillna("NA")
)


# -----------------------------
# Plotly graph
# -----------------------------

# Show taxa with evidence first, but keep no-evidence rows visible if needed
graph_df = plot_df.copy()

# Sort: most problematic/mixed first, then by occurrence
graph_df = graph_df.sort_values(
    ["is_taxid_mixed", "no_of_occurance"],
    ascending=[False, False]
)

fig = px.scatter(
    graph_df,
    x="no_of_occurance",
    y="taxon_label",
    color="resolution_status",
    size="number_of_consensus_taxids",
    hover_name="latin_name",
    hover_data={
        "taxid": True,
        "german_name": True,
        "no_of_occurance": True,
        "LCA_which resolved_ to_this_taxid": True,
        "db_count": True,
        "consensus_taxid": True,
        "consensus_name": True,
        "consensus_rank": True,
        "name_of_the_lca_taxid": True,
        "name_of_the_lca_rank": True,
        "number_of_consensus_taxids": True,
        "number_of_consensus_ranks": True,
        "is_taxid_mixed": True,
        "taxon_label": False,
    },
    title="Taxon resolution status: full, partial, mixed, and higher-rank collapse",
    labels={
        "no_of_occurance": "Predicted amplicon/accession count",
        "taxon_label": "Taxon",
        "resolution_status": "Resolution status",
        "number_of_consensus_taxids": "Number of mixed LCA taxids",
    },
)

fig.update_layout(
    height=1200,
    yaxis={"categoryorder": "total ascending"},
    legend_title_text="Resolution status",
)

fig.update_traces(
    marker=dict(
        opacity=0.8,
        line=dict(width=0.5)
    )
)

fig.show()

NameError: name 'plot_df' is not defined

Issues:
1. when cluster consensus assigns a rank genus if the cluster was not able to successfully resolve to species the Db_count ( is the no of the sequnces in our blast DB ) to be zero as these are species in the db 

In [ ]:
primer_summary = pd.DataFrame([{
    "primer": "ITS2_collapsed",
    "database": "custom",
    "total_clusters": df["cluster_id"].nunique(),
    "total_rows": len(df),
    "unique_accession_taxids": df["accession_taxid"].nunique(),
    "unique_assigned_taxids": df["assigned_taxid"].nunique(),
    "species_level_assignments": (df["assigned_rank"] == "species").sum(),
    "genus_level_assignments": (df["assigned_rank"] == "genus").sum(),
    "family_level_assignments": (df["assigned_rank"] == "family").sum(),
    "varietas_level_assignments": (df["assigned_rank"] == "varietas").sum(),
}])

primer_summary

In [ ]:
taxids_selected = (
    pd.to_numeric(teeliste_df["taxid"].astype(str).str.strip(), errors="coerce")
    .dropna()
    .astype(int)
    .head(74)
)


In [ ]:
def get_lca_from_taxids(taxids):
    taxids_clean = [
        int(str(t).strip())
        for t in taxids
        if str(t).strip().isdigit()
    ]

    if not taxids_clean:
        return None, None, None

    lineages = []

    for taxid in taxids_clean:
        try:
            lineage = ncbi.get_lineage(taxid)
            lineages.append(lineage)
        except Exception:
            print(f"Skipping invalid taxid: {taxid}")

    if not lineages:
        return None, None, None

    common_taxids = set(lineages[0])

    for lineage in lineages[1:]:
        common_taxids = common_taxids.intersection(lineage)

    lca_taxid = None

    for taxid in reversed(lineages[0]):
        if taxid in common_taxids:
            lca_taxid = taxid
            break

    lca_name = ncbi.get_taxid_translator([lca_taxid]).get(lca_taxid, "unknown")
    lca_rank = ncbi.get_rank([lca_taxid]).get(lca_taxid, "unknown")

    return lca_taxid, lca_name, lca_rank

In [ ]:
lca_taxid, lca_name, lca_rank = get_lca_from_taxids(taxids)

print("LCA taxid:", lca_taxid)
print("LCA name:", lca_name)
print("LCA rank:", lca_rank)

In [ ]:


results_df.to_excel("primer_results.xlsx", index=False)

In [ ]:
!pip install plotly
import pandas as pd
import plotly.express as px

plot_df = results_df.copy()

# Remove accidental header row
plot_df = plot_df[
    pd.to_numeric(plot_df["taxid"], errors="coerce").notna()
].copy()

# Convert numeric columns
numeric_cols = [
    "no_of_occurance",
    "LCA_which resolved_ to_this_taxid",
    "db_count",
    "count",
    "min",
    "mean",
    "median",
    "max",
]

for col in numeric_cols:
    plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")


def classify_result(row):
    if row["no_of_occurance"] > 0 and row["LCA_which resolved_ to_this_taxid"] > 0:
        return "Amplifies + species resolved"

    if row["no_of_occurance"] > 0 and row["LCA_which resolved_ to_this_taxid"] == 0:
        return "Amplifies but collapsed"

    if row["no_of_occurance"] == 0 and row["LCA_which resolved_ to_this_taxid"] > 0:
        return "Higher-rank signal only"

    return "No predicted amplicon"


plot_df["interpretation"] = plot_df.apply(classify_result, axis=1)

plot_df["length_range"] = plot_df["max"] - plot_df["min"]

In [ ]:
status_counts = (
    plot_df["interpretation"]
    .value_counts()
    .reset_index()
)

status_counts.columns = ["interpretation", "taxa_count"]

fig = px.bar(
    status_counts,
    x="taxa_count",
    y="interpretation",
    orientation="h",
    title="Primer amplification and taxonomic resolution summary",
    labels={
        "taxa_count": "Number of taxa",
        "interpretation": "Interpretation",
    },
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [ ]:
top_amp = (
    plot_df[plot_df["no_of_occurance"] > 0]
    .sort_values("no_of_occurance", ascending=False)
    
)

fig = px.bar(
    top_amp,
    x="no_of_occurance",
    y="latin_name",
    orientation="h",
    color="interpretation",
    title="Top taxa predicted to amplify",
    labels={
        "no_of_occurance": "Predicted amplicon/accession count",
        "latin_name": "Taxon",
        "interpretation": "Result",
    },
    hover_data=[
        "taxid",
        "german_name",
        "name_of_the_lca_taxid",
        "consensus_name",
        "consensus_rank",
        "mean",
        "amplicon_lengths",
    ],
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [ ]:
length_df = plot_df[
    plot_df["mean"].notna() &
    plot_df["min"].notna() &
    plot_df["max"].notna()
].copy()

length_df = length_df.sort_values("mean")

fig = px.scatter(
    length_df,
    x="mean",
    y="latin_name",
    color="interpretation",
    error_x=length_df["max"] - length_df["mean"],
    error_x_minus=length_df["mean"] - length_df["min"],
    title="Amplicon length range by taxon",
    labels={
        "mean": "Mean amplicon length, bp",
        "latin_name": "Taxon",
        "interpretation": "Result",
    },
    hover_data=[
        "taxid",
        "german_name",
        "count",
        "min",
        "median",
        "max",
        "amplicon_lengths",
        "consensus_name",
        "consensus_rank",
    ],
)

fig.update_layout(height=1200)
fig.show()

In [ ]:
collapsed = plot_df[
    (plot_df["no_of_occurance"] > 0) &
    (plot_df["LCA_which resolved_ to_this_taxid"] == 0)
].copy()

collapsed_rank_df = (
    collapsed.assign(
        collapsed_rank=collapsed["consensus_rank"].astype(str).str.split(";")
    )
    .explode("collapsed_rank")
)

collapsed_rank_counts = (
    collapsed_rank_df["collapsed_rank"]
    .value_counts()
    .reset_index()
)

collapsed_rank_counts.columns = ["collapsed_rank", "taxa_count"]

fig = px.bar(
    collapsed_rank_counts,
    x="taxa_count",
    y="collapsed_rank",
    orientation="h",
    title="Where unresolved taxa collapse after consensus",
    labels={
        "taxa_count": "Number of taxa",
        "collapsed_rank": "Collapsed rank",
    },
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [ ]:
bias_df = plot_df[
    plot_df["db_count"].notna() &
    plot_df["no_of_occurance"].notna() &
    (plot_df["db_count"] > 0)
].copy()

fig = px.scatter(
    bias_df,
    x="db_count",
    y="no_of_occurance",
    color="interpretation",
    hover_name="latin_name",
    title="Database representation vs primer recovery",
    labels={
        "db_count": "Database count",
        "no_of_occurance": "Predicted amplicon/accession count",
        "interpretation": "Result",
    },
    hover_data=[
        "taxid",
        "german_name",
        "name_of_the_lca_taxid",
        "consensus_name",
        "consensus_rank",
    ],
    log_x=True,
)

fig.show()